In [5]:
import pandas as pd

# بارگذاری فایل CSV
file_path = r'C:\Users\Lenovo\Desktop\merged_file.csv'   # مسیر فایل CSV خود را اینجا قرار دهید
df = pd.read_csv(file_path)

# لیستی برای ذخیره نتایج
results = []

# برای هر عدد در ستون Run
for run in df['run'].unique():
    # فیلتر کردن دادهها بر اساس مقدار Run
    run_data = df[df['run'] == run]
    
    # برای هر Residue Number
    for residue_number in run_data['Residue Number'].unique():
        # فیلتر کردن مقادیر Moving Average کمتر از -0.1
        filtered_data = run_data[(run_data['Residue Number'] == residue_number) & ((run_data['Moving Average'] < -0.1))]
        
        if not filtered_data.empty:
            # محاسبه میانگین برای Moving Average و سایر ستونها
            moving_average_mean = filtered_data['Moving Average'].mean()
            internal_mean = filtered_data['Internal'].mean()
            vdW_mean = filtered_data['van der Waals'].mean()
            electrostatic_mean = filtered_data['Electrostatic'].mean()
            polar_solvation_mean = filtered_data['Polar Solvation'].mean()
            non_polar_solvation_mean = filtered_data['Non-Polar Solv.'].mean()
            total_mean = filtered_data['TOTAL'].mean()
            unnamed=filtered_data['Unnamed: 0']
            Residue= filtered_data['Residue']
            
            # ذخیره فقط یک نتیجه برای هر Residue Number و Run
            results.append({
                'Residue Number': residue_number,
                'Residue': Residue,
                'Run': run,
                'Moving Average Mean': moving_average_mean,
                'Internal Mean': internal_mean,
                'van der Waals Mean': vdW_mean,
                'Electrostatic Mean': electrostatic_mean,
                'Polar Solvation Mean': polar_solvation_mean,
                'Non-Polar Solv. Mean': non_polar_solvation_mean,
                'TOTAL Mean': total_mean
            })


results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='Moving Average Mean')
results_df.to_csv('neg_averages_output_separated_runs.csv', index=False)

In [37]:
import pandas as pd
# میانگین همه ران ها

file_path = 'merged_file.csv'  
df = pd.read_csv(file_path)
results = []

for residue_number in df['Residue Number'].unique():

    filtered_data = df[(df['Residue Number'] == residue_number) & (df['Moving Average'] < -0.1)]
    
    if not filtered_data.empty:
        moving_average_mean = filtered_data['Moving Average'].mean()
        internal_mean = filtered_data['Internal'].mean()
        vdW_mean = filtered_data['van der Waals'].mean()
        electrostatic_mean = filtered_data['Electrostatic'].mean()
        polar_solvation_mean = filtered_data['Polar Solvation'].mean()
        non_polar_solvation_mean = filtered_data['Non-Polar Solv.'].mean()
        total_mean = filtered_data['TOTAL'].mean()
        unnamed=filtered_data['Unnamed: 0']
        Residue= filtered_data['Residue']

        
        # ذخیره نتیجه  
        results.append({
            'Residue Number': residue_number,
            'Residue': Residue, 
            'Moving Average Mean': moving_average_mean,
            'Internal Mean': internal_mean,
            'van der Waals Mean': vdW_mean,
            'Electrostatic Mean': electrostatic_mean,
            'Polar Solvation Mean': polar_solvation_mean,
            'Non-Polar Solv. Mean': non_polar_solvation_mean,
            'TOTAL Mean': total_mean})



results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='Moving Average Mean')
results_df.to_csv('averages_all_runs.csv', index=False)

In [9]:
import pandas as pd
import numpy as np

# بارگذاری داده‌ها از فایل CSV
df = pd.read_csv(r'C:\Users\Lenovo\Desktop\merged_file.csv')

# انتخاب ستون‌های مورد نیاز
df = df[['Residue Number', 'run', 'Moving Average', 'Residue', 
         'Internal', 'van der Waals', 'Electrostatic', 
         'Polar Solvation', 'Non-Polar Solv.', 'TOTAL']]

# ایجاد یک DataFrame خالی برای ذخیره نتایج
results = []

# حلقه بر روی هر عدد Run
for run in df['run'].unique():
    run_df = df[df['run'] == run]
    
    # گروه‌بندی بر اساس Residue Number
    for residue_number in run_df['Residue Number'].unique():
        residue_df = run_df[run_df['Residue Number'] == residue_number]
        
        # فیلتر کردن Moving Average کمتر از -0.1
        moving_avg_filtered = residue_df[(residue_df['Moving Average'] < -0.1)|(residue_df['Moving Average'] >0.1)]
        
        if not moving_avg_filtered.empty:
            # پیدا کردن فریم‌های نزدیک و کمترین Moving Average
            min_moving_avg = moving_avg_filtered['Moving Average'].min()
            min_rows = moving_avg_filtered[moving_avg_filtered['Moving Average'] == min_moving_avg]
            
            # محاسبه میانگین برای ستون‌های دیگر
            mean_values = min_rows[['Internal', 'van der Waals', 
                                    'Electrostatic', 'Polar Solvation', 
                                    'Non-Polar Solv.', 'TOTAL']].mean()
            
            # تعداد مقادیر
            count_values = len(min_rows)
            
            # ذخیره نتایج
            result_row = {
                'Residue Number': residue_number,
                'Run': run,
                'Residue': min_rows['Residue'].iloc[0],  # فقط یک Residue ذخیره می‌شود
                'Min Moving Average': min_moving_avg,
                'Count': count_values,
                **mean_values.to_dict()
            }
            results.append(result_row)

# تبدیل نتایج به DataFrame
results_df = pd.DataFrame(results)

# مرتب‌سازی بر اساس Min Moving Average
results_df.sort_values(by='Min Moving Average', inplace=True)

# ذخیره نتایج در فایل CSV
results_df.to_csv('output.csv', index=False)

# ذخیره اطلاعات مربوط به Residue Number, Run و Min Moving Average در فایل جداگانه
residue_info_df = results_df[['Residue Number', 'Run', 'Min Moving Average', 'Residue']]
residue_info_df.to_csv('residue_info.csv', index=False)